# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [ ]:
TABLE = 'support_tickets'
SEGMENT_COLUMN = 'category'
SEGMENT = 'кэшбэк'
PERIOD_START = '2025-04-01'
PERIOD_END = '2025-09-30'

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
    
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [ ]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [ ]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [ ]:
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [ ]:
def task1(rows: list[dict]) -> list:
    total_rows = len(rows)
    total_ok = sum(1 for row in rows if row['ok'])
    total_value = sum(
        (row['value'] for row in rows if row['ok'] and row['value'] is not None),
        Decimal('0')
        )
    
    value_float= float(round(total_value,2))
    unique_days = len(set(row['day'] for row in rows))
    return [total_rows, total_ok, value_float, unique_days]

answer_1 = task1(rows)
answer_1

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [ ]:
from decimal import Decimal
total_rows = answer_1[0]
total_ok = answer_1[1]
total_not_ok = total_rows - total_ok
share_not_ok = (total_not_ok / total_rows) * 100 if total_rows > 0 else 0
not_ok_value_sum = sum(
    (row['value'] for row in rows if not row['ok'] and row['value'] is not None),
    Decimal('0')
    )

print(f"Всего строк: {total_rows}")
print(f"Не состоялось (ok = False): {total_not_ok} строк ({share_not_ok:.1f}%)")
print(f"Сумма по нерешенным (value): {float(not_ok_value_sum)}")




#### ✍️ Ответ на вопрос 1

Не состоялось 7.8% выгрузки (доля строк), это строки с ok = False — 79 из 1015. Это значит, что каждое примерно тринадцатое событие не состоялось. Vakue по данным строкам = 0.0, так как у несостоявшихся событий данное поле не заполнено (None), поэтому тут важен показатель количества и доля строк.


### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [ ]:
def task2(rows: list[dict]) -> int:
    actor_ok = set(row['actor_id'] for row in rows if row['ok'])
    actor_not_ok = set(row['actor_id'] for row in rows if not row['ok'])
    both = (len(actor_ok & actor_not_ok))
    return both

answer_2 = task2(rows)
answer_2
print(f'Количество участников с "ok = True" и "ok = False": {answer_2}')


**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [ ]:
total_rows = len(rows)
unique_actors = len(set(row['actor_id'] for row in rows ))
avg_rows = total_rows / unique_actors

print(f"Всего строк: {total_rows}")
print(f"Количество уникальных участников: {unique_actors}")
print(f"Среднее количество строк на участника: {avg_rows}")
print(f"Количество участников с обоими исходами: {answer_2}")

#### ✍️ Ответ на вопрос 2

На одного участника в среднем приходится: 1.015 строк, почти ровно одна строка на участника. У ответа answer_2 показатель 0, для иного исхода требуется 2 строки на участника, что практически не встречается в таблице. Сегменты "строка" и "участик" дают одинаковый результат, следовательно они неразлечимы.

### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник

def task3(rows: list[dict]) -> dict:
    sum_per_weekday = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            dayIndex = row["day"].weekday()

            row_weekday = WEEKDAYS[dayIndex]
            
            sum_per_weekday[row_weekday] += row["value"]
    return {day: float(round(total, 2)) for day, total in sum_per_weekday.items()}

answer_3 = task3(rows)
answer_3

print(f"Сумма по дням недели:")
for day, total in answer_3.items():
    print(f"{day}: {total}")


**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [ ]:
def task3_1(rows: list[dict]) -> dict:
    dates_per_weekday = defaultdict(set)  
    for row in rows:
        if row['ok']:
            day_name = WEEKDAYS[row['day'].weekday()] 
            dates_per_weekday[day_name].add(row['day'])
    return {day: len(dates) for day, dates in dates_per_weekday.items()}

answer_3_1 = task3_1(rows)
print("Количество различных дат по дням недели:")
for day, count in answer_3_1.items():
    print(f"{day}: {count}")

def task3_2(answer_3: dict, answer_3_1: dict) -> dict:
    avg_per_day = {}
    for weekday in WEEKDAYS:
        if (weekday in answer_3) and (answer_3_1.get(weekday, 0) > 0):
            avg_per_day[weekday] = round(answer_3[weekday] / answer_3_1[weekday], 2)
    return avg_per_day

answer_3_2 = task3_2(answer_3, answer_3_1)
print(f"Средняя сумма на одну дату по дням недели:")
for day, avg in answer_3_2.items():
    print(f"{day}: {avg}")

leader_by_sum = max(answer_3.items(), key=lambda x: x[1])
leader_by_avg = max(answer_3_2.items(), key=lambda x: x[1])

print(f"Лидер по сумме: {leader_by_sum[0]} ({leader_by_sum[1]})")
print(f"Лидер по средней сумме на дату: {leader_by_avg[0]} ({leader_by_avg[1]})")

#### ✍️ Ответ на вопрос 3

Количество различных дат по дням недели: варьируется от 25 до 27. Лидер по сумме и по средней сумме на дату по дням недели - Mon. Данные показатели совплаи из-за небольшого разброса дат, а итог показал, что каждая дата понедельника в среднем дает больший показатель, чем даты других дней.

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [ ]:
def task4(rows):
    result_counter = Counter(row['actor_id'] for row in rows if row['ok'])
    max_counter_value = max(result_counter.values()) 
    winner_ids = [actor_id for actor_id, count in result_counter.items() if count == max_counter_value]
    min_id = min(winner_ids)
    return [min_id, max_counter_value]

answer_4 = task4(rows)
answer_4

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [ ]:
result_counter = Counter(row['actor_id'] for row in rows if row['ok'])
max_counter_value = max(result_counter.values())

winner_ids = [actor_id for actor_id, count in result_counter.items() if count == max_counter_value]

print(f"Максимальное число: {max_counter_value}")
print(f"Число участников с этим числом (делят 1-е место): {len(winner_ids)}")
print(f"Их actor_id: {winner_ids}")
print(f"Min по строке: {answer_4}")



#### ✍️ Ответ на вопрос 4

Первое место делят 15 участников. Победитель был выбран благодаря правилу, так как для выбора среди одинаковых числовых показателей мы привлекли еще один параметр - меньший actor_id при сравнении. Маркетингу не стоит вручать приз по данному значению, так как оно не оражает статус "самый активный", а выбирает среди 15 активных того, чей actor_id меньше.

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [ ]:
def task5(rows: list[dict]) -> list:
    sum_per_day = defaultdict(Decimal)
    for row in rows:
        if row['ok']:
            sum_per_day[row['day']] += row['value']
    sorted_days = sorted(sum_per_day.items(), key=lambda x: (-x[1], x[0]))
    top3 = [[day.isoformat(), float(round(total, 2))] for day, total in sorted_days[:3]]
    return top3

answer_5 = task5(rows)
answer_5

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [ ]:
WEEKDAYS_RU = ["понедельник", "вторник", "среда", "четверг", "пятница", "суббота", "воскресенье"]
for day_str, total in answer_5:
    d = date.fromisoformat(day_str)
    weekday_idx = d.weekday()
    print(f"{day_str}: {WEEKDAYS_RU[weekday_idx]} ({WEEKDAYS_RU[weekday_idx].capitalize()}), сумма = {total}")

weekend_count = sum(
    1 for day_str, _ in answer_5
    if date.fromisoformat(day_str).weekday() in (5, 6) 
)

print(f"Среди трех пиковых дней — суббот и воскресений: {weekend_count}")


#### ✍️ Ответ на вопрос 5

3 пиковых дня: ['2025-06-10', 211.85], ['2025-09-28', 194.94], ['2025-09-08', 188.69], сб - 0, вс - 1.
Гипотеза: пик пришелся на эти даты по причине каких-то событий (мероприятий, акций и т.д.), а не из-за дня недели, иначе сб входила бы в пик и скорее всего было бы пт, сб и вс, а не пн, вт, вс.
Проверить: посчитать среднюю сумму value на другие даты этих же дней (например: взять 5 вторников и высчитать средний показатель по разным датам) и сравнить средние показатели этих дней с их рекордными. Так же стоит посмотреть на количество строк ok = True на пиковые даты, если сумма большая, а количество строк небольшое — скорее всего это выброс. Если строк тоже много, то причина пиков скорее всего кроется в определенных событиях этих дат (мероприятий, акции). 

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [ ]:
def task6(rows: list[dict]) -> list:
    sums = defaultdict(Decimal)
    for row in rows:
        if row['ok']:
            sums[row['day']] += row['value']
    total = sum(sums.values())
    half = total / 2

    running_total = Decimal(0)
    for number, day in enumerate(sorted(sums), start=1):
        running_total += sums[day]
        if running_total >= half:
            return [number, day.isoformat()]
        
    return None


answer_6 = task6(rows)
answer_6

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [ ]:

import math

sums = defaultdict(Decimal)
for row in rows:
    if row['ok']:
        sums[row['day']] += row['value']

total_days = len(sums)
day_number = answer_6[0]
day_date = answer_6[1]

fraction = day_number / total_days
expected_number = math.ceil(total_days / 2)
expected_fraction = expected_number / total_days

print(f"Всего дней в окне (с хотя бы одной строкой ok=True): {total_days}")
print(f"Номер дня, на котором набежала половина суммы: {day_number} ({day_date})")
print(f"Фактическая доля: {day_number}/{total_days} = {fraction:.3f} ({fraction*100:.1f}%)")
print(f"Ожидаемый номер при равномерном накоплении: {expected_number} (округление вверх от половины)")
print(f"Ожидаемая доля: {expected_number}/{total_days} = {expected_fraction:.3f} ({expected_fraction*100:.1f}%)")
print(f"Разница: день {day_number} против ожидаемых {expected_number} — {'раньше' if day_number < expected_number else ('позже' if day_number > expected_number else 'ровно как ожидалось')}")

#### ✍️ Ответ на вопрос 6

Первая половина суммы заняла: 93/181 = 0.514 (51.4%), при равномерном накоплении ожидалось: 91/181 = 0.503 (50.3%). Разница в 2 дня показывает нам, что накопление суммы шло почти равномерно, без явных перекосов. Да, произошло смешение на 2 дня, что может свидетельствовать о чуть более высокой активности во второй половине, но не стоит говорить о разгоне или торможении участка, он был стабильным. 


### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [ ]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]


assert RAW[1][1] == " " and RAW[5][1] == "\u00a0"
def to_value(x: str | None) -> float | None:
    if x is None:
        return None

    x = x.strip()
    if x == "" or x == "—":
        return None

    cleaned = "".join(x.split())
    cleaned = cleaned.replace(",", ".")

    try:
        return float(cleaned)
    except ValueError:
        return None

def task7(raw: list) -> list:
    total = 0.0
    dropped = 0
    for x in raw:
        value = to_value(x)
        if value is None:
            dropped += 1
        else:
            total += value
    return [round(total, 2), dropped]


answer_7 = task7(RAW)
answer_7

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [ ]:
total_values = len(RAW)
total_sum, dropped = answer_7
parsed = total_values - dropped
share_dropped = dropped / total_values

print(f"Всего значений в RAW: {total_values}")
print(f"Разобрано успешно: {parsed}")
print(f"Отброшено (потеряно): {dropped}")
print(f"Доля потерянных значений: {share_dropped:.1%}")
print(f"Итоговая сумма по разобранным значениям: {total_sum}")

#### ✍️ Ответ на вопрос 7

Опаснее - молчаливый возврат "None", что занижает итоговую сумму и не подсвечивает ошибку. В данном случае из 8 значений RAW успешно разобрано только 5, отброшено 3 (37.5%), а итоговая сумма: 4237.5 - выглядит как корректный итог, но она занижена. Руководителю направила бы всю вышеупомянутую информацию и список нераспознанных строк (для устранения причины). 


### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [ ]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}


def task8(rows: list[dict]) -> float:
    sum_per_weekday = defaultdict(Decimal)
    for row in rows:
        if row['ok']:
            day_name = WEEKDAYS[row['day'].weekday()]
            sum_per_weekday[day_name] += row['value']

    weighted_total = 0.0
    for day_name, total in sum_per_weekday.items():
        weight = WEIGHTS.get(day_name, 1.0)
        weighted_total += float(total) * weight

    return round(weighted_total, 2)


answer_8 = task8(rows)
answer_8

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [ ]:
weekdays_in_data = set()
for row in rows:
    if row['ok']:
        weekdays_in_data.add(WEEKDAYS[row['day'].weekday()])

default_weight_days = sorted(weekdays_in_data - set(WEIGHTS.keys()))
print(f"Дни недели, встретившиеся в данных: {sorted(weekdays_in_data)}")
print(f"Дни, которые пошли с коэффициентом по умолчанию (1.0): {default_weight_days}")

plain_total = sum(row['value'] for row in rows if row['ok'])
plain_total = float(round(plain_total, 2))

weighted_total = answer_8

diff = weighted_total - plain_total
percent_diff = diff / plain_total * 100

print(f"Обычная сумма (задача 1): {plain_total}")
print(f"Взвешенная сумма (задача 8): {weighted_total}")
print(f"Разница: {round(diff, 2)}")
print(f"Отличие в процентах от обычной суммы: {percent_diff:.1f}%")

#### ✍️ Ответ на вопрос 8

Дни, которые пошли с коэффициентом по умолчанию (1.0): 'Sat', 'Sun'. Разница взвешенной суммы от обычной: 34.24 (0.4%). Ситуация на данных ПРАЙМ, при которой такой .get спрятал бы ошибку: в справочнике WEIGHTS была опечатка в названии дня или аналитик спустя время добавил бы коэффициенты для выходных, но забыл бы обновить справочник в коде.

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [ ]:
# Исследование А: ваш код. Всё — от значений ячейки 1.1.

import statistics

full_sql = """
    select
        count(distinct {actor}::text) as n_actors,
        avg(case when {ok} then 1.0 else 0.0 end) as ok_share,
        sum(case when {ok} then {value} else 0 end)::float
            / count(distinct {actor}::text) as value_per_actor
    from prime.{table}
    where {segment_column} = :segment
      and {moment} >= cast(:period_start as date)
      and {moment} <  cast(:period_end as date) + 1
""".format(table=TABLE, **COLUMNS[TABLE])

full_stats = query(full_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

full_row = full_stats[0]
n_actors_full = full_row['n_actors']
ok_share_full = float(full_row['ok_share'])
value_per_actor_full = float(full_row['value_per_actor'])

print("Весь сегмент:", n_actors_full, ok_share_full, value_per_actor_full)

def stats_from_rows(data):
    n_actors = len(set(r['actor_id'] for r in data))
    ok_share = sum(1 for r in data if r['ok']) / len(data) if len(data) > 0 else 0.0
    value_per_actor = sum(r['value'] for r in data if r['ok']) / n_actors
    return n_actors, float(ok_share), float(value_per_actor)

my_thousand = stats_from_rows(rows)
print("Моя тысяча:", my_thousand)

salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])

samples = []
for i in range(25):
    rows_i = query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))
    samples.append(stats_from_rows(rows_i))

ok_shares = [s[1] for s in samples]
values = [s[2] for s in samples]

mean_value = statistics.mean(values)
sd_value = statistics.pstdev(values)

print("Среднее value/участник по 25 тысячам:",round(mean_value,2))
print("stdev между тысячами (это и есть SE при n=1000):", round(sd_value,2))

relative_diff = (my_thousand[2] - value_per_actor_full) / value_per_actor_full
print("Относительное отличие тысячи от сегмента:", round(relative_diff,2))

shift_5pct = 0.05 * mean_value
print(f"5%-ный сдвиг = {shift_5pct:.4f} ч, stdev между тысячами = {sd_value:.4f} ч")
print("Сдвиг больше шума между тысячами?", shift_5pct > sd_value)

target_se = shift_5pct / 2
required_n = 1000 * (sd_value / target_se) ** 2
print("Нужно участников для надежного обнаружения 5%-го сдвига:", round(required_n))

Вопрос. Насколько можно доверять показателям по тысяче участников, если реальный сегмент — 5888 человек? Будет ли заметно по одной такой тысяче падение value на участника на 5%?

**Как проверяли**. Проверка осуществлялась следующим образом: произвела сравнение одного и того же показателя 3 разными способами. 1. По всем участникам сразу; 2. По своей тысяче; 3. По 25 другим случайным тысячам, посчитала среднее и разброс, чтобы сравнить каждую тысячу с общим средним и сегментом (через стандартное отклонение, stdev).

**Что получилось**. Показатели, на основе которых был сделан вывод:
Весь сегмент: 5888 0.9159410582719357 7.345259850543478
Моя тысяча: (1000, 0.9221674876847291, 7.8782)
Среднее value/участник по 25 тысячам: 7.26
stdev между тысячами (это и есть SE при n=1000): 0.34
Относительное отличие тысячи от сегмента: 0.07
5%-ный сдвиг = 0.3629 ч, stdev между тысячами = 0.3358 ч
Сдвиг больше шума между тысячами? True
Нужно участников для надежного обнаружения 5%-го сдвига: 3424

**Что это значит и чего расчет не доказывает**. Одной тысяче доверять можно, но нужно быть готовым к тому, что она может отличаться от реальности на 5–7% из-за случайности формирования выборки, без изменения в данных. Следовательно, если в следующем месяце мы увидим, например, показатель на 5 процентов ниже — не факт, что что-то действительно изменилось, это может быть просто случайная флуктуация выборки. Сдвиг в 5% (0.36 ч) чуть больше типичного разброса между тысячами (0.34 ч), его можно заметить, но разница между показателями достаточно мала, поэтому его легко спутать с шумом. Для надежности и получения точных данных я бы взяла больше половины участников или весь сегмент целиком.

**Расчет не доказывает**: меняется ли состав активных клиентов от месяца к месяцу, приток/отток клиентов и другие факторы, которые могут менять показатель независимо от случайной выборки.


### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [ ]:
# Исследование Б: ваш код. Всё — от значений ячейки 1.1.

from datetime import timedelta

by_segment_sql = """
    select
        {segment_column}                                            as segment,
        count(distinct {actor}::text)                                as n_actors,
        count(*)::float / count(distinct {actor}::text)               as rows_per_actor,
        avg(case when {ok} then 1.0 else 0.0 end)                     as ok_share,
        sum(case when {ok} then {value} else 0 end)::float
            / nullif(sum(case when {ok} then 1 else 0 end), 0)        as value_per_ok_row,
        sum(case when {ok} then {value} else 0 end)::float
            / count(distinct {actor}::text)                          as value_per_actor
    from prime.{table}
    where {moment} >= cast(:period_start as date)
      and {moment} <  cast(:period_end as date) + 1
    group by {segment_column}
    order by value_per_actor desc
""".format(table=TABLE, **COLUMNS[TABLE])

current_by_segment = query(by_segment_sql, period_start=PERIOD_START, period_end=PERIOD_END)

print("Текущее окно")
print(f"{'segment':<15}{'n_actors':>10}{'rows/act':>10}{'ok_share':>10}{'val/ok_row':>12}{'val/actor':>12}")
for row in current_by_segment:
    mark = " <-- мой" if row['segment'] == SEGMENT else ""
    print(f"{row['segment']:<15}{row['n_actors']:>10}{float(row['rows_per_actor']):>10.2f}"
          f"{float(row['ok_share']):>10.3f}{float(row['value_per_ok_row']):>12.3f}"
          f"{float(row['value_per_actor']):>12.3f}{mark}")

print("\nРазложение value_per_actor на множители")

rows_per_actor_vals = [float(r['rows_per_actor']) for r in current_by_segment]
ok_share_vals = [float(r['ok_share']) for r in current_by_segment]
value_per_ok_row_vals = [float(r['value_per_ok_row']) for r in current_by_segment]

cv_rows = statistics.stdev(rows_per_actor_vals) / statistics.mean(rows_per_actor_vals) if len(rows_per_actor_vals) > 1 else 0
cv_ok = statistics.stdev(ok_share_vals) / statistics.mean(ok_share_vals) if len(ok_share_vals) > 1 else 0
cv_val = statistics.stdev(value_per_ok_row_vals) / statistics.mean(value_per_ok_row_vals) if len(value_per_ok_row_vals) > 1 else 0

print(f"\nКоэффициент вариации (мера различия между сегментами):")
print(f"  rows_per_actor:   {cv_rows:.3f}")
print(f"  ok_share:         {cv_ok:.3f}")
print(f"  value_per_ok_row: {cv_val:.3f}")

max_cv = max(cv_rows, cv_ok, cv_val)
if max_cv == cv_rows:
    main_driver = "rows_per_actor (число строк на участника)"
elif max_cv == cv_ok:
    main_driver = "ok_share (доля успешных строк)"
else:
    main_driver = "value_per_ok_row (значение на одну успешную строку)"

print(f"\nГлавный драйвер различий между сегментами: {main_driver}")

start = date.fromisoformat(PERIOD_START)
end = date.fromisoformat(PERIOD_END)
window_len = (end - start).days + 1

prev_end = start - timedelta(days=1)
prev_start = prev_end - timedelta(days=window_len - 1)

DATA_START = date(2025, 1, 6)
window_is_full = prev_start >= DATA_START

print(f"\nПредыдущее окно: {prev_start} .. {prev_end} (полное: {window_is_full})")

prev_by_segment = query(by_segment_sql, period_start=prev_start.isoformat(), period_end=prev_end.isoformat())

print(f"{'segment':<15}{'n_actors':>10}{'rows/act':>10}{'ok_share':>10}{'val/ok_row':>12}{'val/actor':>12}")
for row in prev_by_segment:
    mark = " <-- мой" if row['segment'] == SEGMENT else ""
    print(f"{row['segment']:<15}{row['n_actors']:>10}{float(row['rows_per_actor']):>10.2f}"
          f"{float(row['ok_share']):>10.3f}{float(row['value_per_ok_row']):>12.3f}"
          f"{float(row['value_per_actor']):>12.3f}{mark}")

my_current = next((r for r in current_by_segment if r['segment'] == SEGMENT), None)
my_prev = next((r for r in prev_by_segment if r['segment'] == SEGMENT), None)

if my_current and my_prev:
    print(f"\nСравнение моего сегмента ({SEGMENT})")
    print(f"{'Метрика':<20}{'Текущее':>12}{'Предыдущее':>12}{'Изменение':>12}")
    
    metrics = [
        ('n_actors', 'n_actors'),
        ('rows_per_actor', 'rows_per_actor'),
        ('ok_share', 'ok_share'),
        ('value_per_ok_row', 'value_per_ok_row'),
        ('value_per_actor', 'value_per_actor')
    ]
    
    for label, key in metrics:
        curr_val = float(my_current[key])
        prev_val = float(my_prev[key])
        change = (curr_val - prev_val) / prev_val if prev_val != 0 else 0
        print(f"{label:<20}{curr_val:>12.3f}{prev_val:>12.3f}{change:>11.1%}")

#### ✍️ Исследование Б

**Вопрос**. Маркетинг хочет перераспределить бюджет в пользу «ценных» сегментов и просит проверить: правда ли сегмент «кэшбэк» отличается от соседних и чем именно.

**Как проверяли**. Произвела подсчет по всем шести сегментам за апрель–сентябрь 2025 года: сколько тикетов на участника, доля решенных, сколько часов уходит на одно решенное обращение и сколько часов на участника всего. Разложила итоговую метрику на множители и рассчитала коэффициент вариации, чтобы объективно найти главный драйвер различий между сегментами. Проверила, повторяется ли это в предыдущем полугодии (учла, что предыдущее окно неполное, начинается раньше даты доступных данных).

**Что получилось**. Самый низкий показатель у «кэшбэка», 7,35 часа на участника, самый высокий у «оплаты», 7,92 часа. Разница в 5–9% почти не связана с количеством обращений. Коэффициент вариации показал, что число тикетов на участника (0,006) и доля успешных решений (0,005) практически идентичны у всех сегментов (около одного тикета на человека, ~92% успеха). Весь разброс итоговой метрики (0,022) обусловлен только множителем «часов на одно решенное обращение». Здесь важна именно скорость решения. Тикеты «кэшбэка» в среднем закрывают быстрее. При этом в предыдущем полугодии сегменты стояли в ином порядке. «Оплата» была почти последней, а сейчас первая. Делаю вывод, что порядок нестабильный. Так как предыдущее окно неполное, абсолютные числа в нем занижены, поэтому сравниваем только относительные доли и средние значения, которые подтверждают эту нестабильность. Сделанные расчеты с пояснениями расположены чуть выше данной ячейки (выведенный результат предыдущей ячейки).

**Что это значит и чего расчет не доказывает**. Value является часами работы поддержки, а не деньгами клиента. Низкий показатель у «кэшбэка» отражает наиболее быстрое решение его тикетов, а не более низкую ценность сегмента. У «оплаты» высокий показатель, что показывает большую нагрузку на поддержку, а не большую ценность клиентов. При этом порядок сегментов от месяца к месяцу сильно скачет, так что пока нельзя точно сказать об устойчивом различии. Расчет не доказывает причинно-следственную связь, то есть почему сегмент ценный, и не доказывает, что увеличение бюджета приведет к пропорциональному росту ценности.

**Различия между сегментами есть, но небольшие и нестабильные**. Важно, что эти данные показывают нагрузку на поддержку, а не выручку. Перераспределять бюджет на их основе нельзя. Чтобы понять какой сегмент реально ценнее, нужно смотреть данные о платежах (payments или transactions), а не о тикетах в поддержку.


## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

_По желанию._